# TEMP — TransNetV2 producer V2 trên Kaggle GPU

Notebook này chỉ dùng để thực thi producer/preflight theo worker; không phải notebook tích hợp/evaluate chính thức.

Thiết kế và contract: [`docs/research/keyframe-augmentation/implementation-v1.md`](../../docs/research/keyframe-augmentation/implementation-v1.md)

**Bấm Run All lần đầu chỉ chạy PREFLIGHT một video ngắn nhất.** Sau khi preflight PASS, đổi `RUN_SCOPE="SHARD"` và `CONFIRM_SHARD=WORKER`; nếu phiên Kaggle đứt thì Run All lại — video có marker và SHA-256 hợp lệ sẽ được bỏ qua.
Lượt equivalence batch `1`/`8` phải dùng `OUTPUT_ROOT_OVERRIDE` khác nhau dưới `/sandbox/`; không chạy benchmark bằng production root.

Notebook này là **producer theo shard**, không phải notebook tune/evaluate. Corpus có **873 video trong 14 ZIP, tổng 77,29 GiB**. Nó tải từng video BTC, đọc literal FPS từ mapping BTC chuẩn, chạy official TransNetV2 đã pin commit rồi đẩy shot proposals + provenance lên Modal Volume nghiên cứu chung. Video được xóa sau mỗi lượt nên không cần chứa cả dataset trên disk Kaggle.

### Bạn chỉ cần sửa hai dòng cho lần preflight

Ở cell Step 0a, đổi `MEMBER` thành tên chủ sở hữu thật (`triet`, `trong`, `quang`, `bao`, `dat`) và đổi `WORKER` thành worker của tài khoản đó. Ví dụ Trọng dùng `MEMBER = "trong"`, `WORKER = "trong1"` hoặc `"trong2"`; Quang dùng `quang1`–`quang4`. **Không đổi** threshold, commit, volume, output root hay assignment.

| người | worker | video/worker | GiB/worker | ZIP gốc |
|---|---:|---:|---:|---|
| triet | 2 | ~59–60 | ~7,92 | L22_a, L25_a |
| trong | 2 | ~81–82 | ~8,02 | L24_a, L28_a, L30_a |
| quang | 4 | ~56–57 | ~3,94 | L21_a, L26_a, L26_e |
| bao | 2 | ~74 | ~7,32 | L23_a, L26_c, L29_a |
| dat | 2 | ~108 | ~7,52 | L26_b, L26_d, L27_a |

Chia theo dung lượng thay vì ép số video bằng nhau vì thời gian tải/giải mã gần với số byte và thời lượng hơn số file. Mỗi worker có slice tất định, disjoint; inventory kiểm đủ **873/873**, không có video trùng, thiếu hoặc nằm ngoài assignment. Không chạy hai phiên cùng `WORKER`.

### Cần có trước khi chạy

| | thiết lập |
|---|---|
| Kaggle Settings | Accelerator = **GPU**, Internet = **On** |
| Kaggle Secrets | `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`; thêm `GITHUB_TOKEN` nếu repo private |
| Repo branch | `feature/quang/storage` phải chứa notebook, config và package `keyframe_augmentation` |

Không dán token vào code hoặc output. Cell GPU sẽ dừng ngay nếu Kaggle chưa cấp CUDA; cell upstream sẽ dừng nếu weight không đến từ repository chính thức.

### Dữ liệu đi đâu?

- `keyframes-btc`: **chỉ đọc** mapping chuẩn, không sửa bundle đối chứng.
- `keyframe-augmentation-research`: ghi `shots/`, `reports/`, `markers/` và worker manifests.
- Git: chỉ chứa notebook/config/code; **không** đưa video, weights hoặc output model vào Git.

Marker `COMPLETE` luôn được upload cuối cùng sau khi đọc lại SHA-256. File upload dở không được tính là hoàn tất.

### Khi nào phần của bạn xong?

Cell cuối phải in `verified_complete == assigned` và `failures_this_session == 0`. Thời gian thực tế phụ thuộc loại GPU, độ dài video và băng thông host; chưa có số đo Kaggle đáng tin cậy, nên xem `elapsed_seconds` của vài video đầu trước khi ước lượng cả shard. Có thể ngắt phiên và chạy lại an toàn.

> Không chạy Gate1 riêng cho shard của bạn. Hard delta budget phải được tính đúng **một lần trên toàn corpus** sau khi đủ output của năm người.

## Step 0a — repo, dependencies và công tắc member

Đổi đúng hai dòng `MEMBER` và `WORKER`, sau đó chạy cell. Cell clone repo mà không in token, cài gói tối thiểu và in commit/config hash để cả nhóm biết mình đang chạy cùng một phiên bản. Nếu rerun trong cùng session, notebook dùng checkout hiện có. `WORKER` là slice disjoint nên hai tài khoản được phép chạy song song mà không đụng marker.

In [ ]:
from __future__ import annotations

import csv
import gc
import json
import math
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# Đổi hai dòng này. MEMBER là owner thật; WORKER là slice/account riêng.
MEMBER = "quang"
WORKER = "quang1"
RUN_SCOPE = "PREFLIGHT"  # mặc định chỉ video pending ngắn nhất; "SHARD" mới chạy tiếp
CONFIRM_SHARD = ""       # khi RUN_SCOPE="SHARD", đặt đúng bằng WORKER
SESSION_MAX_MINUTES = 240   # không khởi động video mới sau mốc này; rerun để resume
INFERENCE_WINDOW_BATCH_SIZE = 8  # performance-only; giảm nếu CUDA OOM
OUTPUT_ROOT_OVERRIDE = None  # chỉ dùng sandbox equivalence; production để None

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/keyframe-augmentation")
WORK_DIR.mkdir(parents=True, exist_ok=True)

def _redact_text(value: object, secret_values: tuple[str, ...]) -> str:
    text = value.decode("utf-8", "replace") if isinstance(value, bytes) else str(value)
    for secret in secret_values:
        if secret:
            text = text.replace(secret, "[REDACTED]")
    return text

def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None, secret_values: tuple[str, ...] = ()) -> str:
    try:
        result = subprocess.run(command, cwd=cwd, env=env, check=True, text=True, capture_output=True)
    except subprocess.CalledProcessError as error:
        safe_command = " ".join(_redact_text(part, secret_values) for part in command)
        detail = "\n".join((_redact_text(error.stdout, secret_values), _redact_text(error.stderr, secret_values)))
        raise RuntimeError(f"Command failed: {safe_command}\n{detail[-2000:]}") from None
    return _redact_text(result.stdout, secret_values).strip()

def clone_repository() -> None:
    if (REPO_DIR / ".git").is_dir():
        print(f"Dùng checkout hiện có: {REPO_DIR}")
        return
    env = os.environ.copy()
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
    askpass: Path | None = None
    try:
        if token:
            askpass = WORK_DIR / "git_askpass.sh"
            askpass.write_text(
                "#!/bin/sh\ncase \"$1\" in *Username*) printf x-access-token ;; *) printf %s \"$GITHUB_TOKEN\" ;; esac\n",
                encoding="utf-8",
            )
            askpass.chmod(0o700)
            env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
        run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)], env=env, secret_values=(token,) if token else ())
    finally:
        if askpass is not None:
            askpass.unlink(missing_ok=True)

clone_repository()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "notebooks/requirements-kaggle-keyframe-augmentation.txt")],
    check=True,
)
sys.path.insert(0, str(REPO_DIR / "src"))

import yaml
from mfe.offline.bundle import BundleLayout
from mfe.offline.keyframe_augmentation.common import (
    atomic_write_json, atomic_write_jsonl, sha256_file, sha256_json,
)
from mfe.offline.keyframe_augmentation.transnet import detect_video, warmup_transnet
from mfe.offline.source import fetch_video, load_catalogue
from mfe.offline.transport import ModalVolumeTransport

CONFIG_PATH = REPO_DIR / "config/keyframe_augmentation_kaggle.yaml"
CFG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
if MEMBER not in CFG["members"]:
    raise ValueError(f"MEMBER={MEMBER!r} không thuộc {CFG['members']}")
WORKERS = CFG["workers"].get(MEMBER, [])
if WORKER not in WORKERS:
    raise ValueError(f"WORKER={WORKER!r} không thuộc {MEMBER}: {WORKERS}")
if RUN_SCOPE not in {"PREFLIGHT", "SHARD"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT hoặc SHARD")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != WORKER:
    raise RuntimeError("Muốn chạy shard phải đặt CONFIRM_SHARD đúng bằng WORKER")
if SESSION_MAX_MINUTES <= 0:
    raise ValueError("SESSION_MAX_MINUTES phải dương")
if INFERENCE_WINDOW_BATCH_SIZE < 1:
    raise ValueError("INFERENCE_WINDOW_BATCH_SIZE phải >= 1")
if OUTPUT_ROOT_OVERRIDE is not None:
    configured_root = CFG["output"]["root"].rstrip("/")
    sandbox_root = str(OUTPUT_ROOT_OVERRIDE).rstrip("/")
    if not sandbox_root.startswith(configured_root + "/sandbox/"):
        raise ValueError("OUTPUT_ROOT_OVERRIDE phải là root con /sandbox/ để tránh ghi production")
    if sandbox_root == configured_root:
        raise ValueError("OUTPUT_ROOT_OVERRIDE không được trùng output root production")
WORKER_INDEX = WORKERS.index(WORKER)
WORKER_COUNT = len(WORKERS)
REPO_COMMIT = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
CONFIG_SHA256 = sha256_file(CONFIG_PATH)
print({"member": MEMBER, "worker": WORKER, "worker_index": WORKER_INDEX, "worker_count": WORKER_COUNT, "repo_commit": REPO_COMMIT, "config_sha256": CONFIG_SHA256})

## Step 0b — kiểm tra GPU và official TransNetV2

Chỉ checkout repository chính thức `soCzech/TransNetV2` tại commit trong config và lấy weight bằng Git LFS từ chính repository đó. Nếu source/weight không đủ provenance, cell dừng; không fallback sang mirror.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Không thấy CUDA. Trong Kaggle Settings, chọn GPU accelerator rồi restart session.")
if CFG["transnetv2"]["device"] != "cuda":
    raise ValueError("Shared contract phải dùng CUDA cho producer Kaggle này")
print({"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda})

TRANSNET_DIR = Path("/kaggle/working/TransNetV2")
tn = CFG["transnetv2"]
transnet_env = os.environ.copy()
transnet_env["GIT_LFS_SKIP_SMUDGE"] = "1"
if not (TRANSNET_DIR / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", "--no-checkout", tn["repository"], str(TRANSNET_DIR)], env=transnet_env)
run(["git", "fetch", "--depth", "1", "origin", tn["commit"]], cwd=TRANSNET_DIR, env=transnet_env)
run(["git", "checkout", "--detach", tn["commit"]], cwd=TRANSNET_DIR, env=transnet_env)
actual_transnet_commit = run(["git", "rev-parse", "HEAD"], cwd=TRANSNET_DIR)
if actual_transnet_commit != tn["commit"]:
    raise RuntimeError(f"Sai TransNetV2 commit: {actual_transnet_commit}")
run(["git", "lfs", "install", "--local"], cwd=TRANSNET_DIR)
run(["git", "lfs", "pull", "--include", tn["weights_relpath"], "--exclude", ""], cwd=TRANSNET_DIR)
TRANSNET_WEIGHTS = TRANSNET_DIR / tn["weights_relpath"]
TRANSNET_SOURCE = TRANSNET_DIR / tn["source_relpath"]
if not TRANSNET_SOURCE.is_file() or not TRANSNET_WEIGHTS.is_file() or TRANSNET_WEIGHTS.stat().st_size < 1_000_000:
    raise RuntimeError("Thiếu official TransNetV2 source/weight; không dùng mirror thay thế")
TRANSNET_SOURCE_SHA256 = sha256_file(TRANSNET_SOURCE)
TRANSNET_WEIGHTS_SHA256 = sha256_file(TRANSNET_WEIGHTS)
print({"transnet_commit": actual_transnet_commit, "source_sha256": TRANSNET_SOURCE_SHA256, "weights_sha256": TRANSNET_WEIGHTS_SHA256})
TRANSNET_RUNTIME_PREFLIGHT = warmup_transnet(
    repo=TRANSNET_DIR,
    weights=TRANSNET_WEIGHTS,
    device=tn["device"],
    window_batch_size=INFERENCE_WINDOW_BATCH_SIZE,
)
print({"transnet_runtime_preflight": TRANSNET_RUNTIME_PREFLIGHT})

## Step 0c — Modal, phân công và hai volume tách biệt

Assignment dùng đúng `config/zips_batch1.yaml`. `keyframes-btc` được mở với `create_if_missing=False`; output volume nghiên cứu là volume khác. Video được tải từng file bằng HTTP Range và xóa sau khi xử lý để giữ disk peak thấp.

In [ ]:
CATALOGUE_PATH = REPO_DIR / CFG["source"]["catalogue"]
book = yaml.safe_load(CATALOGUE_PATH.read_text(encoding="utf-8"))
assigned_zips = book[CFG["source"]["assignment_key"]][MEMBER]
owner_videos = sorted(video for zip_name in assigned_zips for video in book["zips"][zip_name]["videos"])
catalogue = load_catalogue(CATALOGUE_PATH)
missing_catalogue = sorted(set(owner_videos) - set(catalogue))
if missing_catalogue:
    raise RuntimeError(f"Assignment có video thiếu catalogue: {missing_catalogue[:5]}")

def byte_balanced_slices(video_ids: list[str], count: int) -> list[list[str]]:
    buckets: list[list[str]] = [[] for _ in range(count)]
    totals = [0] * count
    sizes = {video_id: catalogue[video_id].size for video_id in video_ids}
    for video_id in sorted(video_ids, key=lambda value: (-sizes[value], value)):
        index = min(range(count), key=lambda value: (totals[value], value))
        buckets[index].append(video_id)
        totals[index] += sizes[video_id]
    return [sorted(bucket) for bucket in buckets]

worker_slices = byte_balanced_slices(owner_videos, WORKER_COUNT)
assigned_videos = worker_slices[WORKER_INDEX]
if set().union(*map(set, worker_slices)) != set(owner_videos) or sum(map(len, worker_slices)) != len(set(owner_videos)):
    raise AssertionError("Worker slices không disjoint/không phủ đủ owner videos")

btc_transport = ModalVolumeTransport(CFG["btc"]["volume"], create_if_missing=False)
output_transport = ModalVolumeTransport(CFG["output"]["volume"], create_if_missing=True, version=2)
btc_layout = BundleLayout(CFG["btc"]["bundle"])
print({"member": MEMBER, "worker": WORKER, "zips": assigned_zips, "owner_videos": len(owner_videos), "worker_videos": len(assigned_videos), "btc_volume_read_only": CFG['btc']['volume'], "output_volume": CFG['output']['volume']})

In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def remote_paths(video_id: str) -> dict[str, str]:
    root = (OUTPUT_ROOT_OVERRIDE or CFG["output"]["root"]).rstrip("/")
    return {
        "shots": f"{root}/shots/{video_id}.jsonl",
        "report": f"{root}/reports/{video_id}.json",
        "marker": f"{root}/markers/{MEMBER}/{WORKER}/{video_id}.json",
    }

def read_literal_fps(mapping_path: Path) -> tuple[float, int]:
    with mapping_path.open("r", encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows:
        raise ValueError(f"Mapping rỗng: {mapping_path}")
    fps_values: set[float] = set()
    for line_no, row in enumerate(rows, 2):
        fps = float(row["fps"])
        if not math.isfinite(fps) or fps <= 0:
            raise ValueError(f"FPS không hợp lệ tại dòng {line_no}: {fps}")
        fps_values.add(fps)
        raw_idx = (row.get("frame_idx") or "").strip()
        if raw_idx and math.floor(float(row["pts_time"]) * fps) != int(raw_idx):
            raise ValueError(f"Mapping vi phạm floor(pts_time*fps) tại dòng {line_no}")
    if len(fps_values) != 1:
        raise ValueError(f"Mapping có nhiều literal FPS: {sorted(fps_values)}")
    return next(iter(fps_values)), len(rows)

def completion_marker(video_id: str) -> dict | None:
    paths = remote_paths(video_id)
    if not output_transport.exists(paths["marker"]):
        return None
    local = WORK_DIR / "status" / MEMBER / WORKER / f"{video_id}.json"
    try:
        output_transport.get(paths["marker"], local)
        marker = json.loads(local.read_text(encoding="utf-8"))
        if marker.get("status") != "COMPLETE" or marker.get("config_sha256") != CONFIG_SHA256:
            return None
        if marker.get("run_id") != CFG["run_id"] or marker.get("member") != MEMBER or marker.get("video_id") != video_id:
            return None
        marker_payload_hash = marker.get("marker_payload_hash")
        unsigned_marker = dict(marker)
        unsigned_marker.pop("marker_payload_hash", None)
        if not marker_payload_hash or sha256_json(unsigned_marker) != marker_payload_hash:
            return None
        if marker.get("worker") != WORKER or marker.get("transnet_commit") != tn["commit"]:
            return None
        if marker.get("transnet_source_sha256") != TRANSNET_SOURCE_SHA256 or marker.get("transnet_weights_sha256") != TRANSNET_WEIGHTS_SHA256:
            return None
        shots_sha256 = output_transport.sha256(paths["shots"])
        if not marker.get("shots_sha256") or shots_sha256 != marker.get("shots_sha256"):
            return None
        report_sha256 = output_transport.sha256(paths["report"])
        if not marker.get("report_sha256") or report_sha256 != marker.get("report_sha256"):
            return None
        return marker
    except Exception:
        return None
    finally:
        local.unlink(missing_ok=True)

def build_attached_video_index() -> dict[str, list[Path]]:
    root = Path("/kaggle/input")
    if not root.is_dir():
        return {}
    index: dict[str, list[Path]] = {}
    for path in root.rglob("*.mp4"):
        index.setdefault(path.stem, []).append(path)
    return {video_id: sorted(paths) for video_id, paths in index.items()}

ATTACHED_VIDEO_INDEX = build_attached_video_index()

def resolve_video_source(video_id: str, local_video: Path) -> tuple[Path, str]:
    candidates = ATTACHED_VIDEO_INDEX.get(video_id, [])
    expected_size = catalogue[video_id].size
    exact = [path for path in candidates if path.stat().st_size == expected_size]
    if len(exact) > 1:
        raise RuntimeError(f"Có nhiều attached MP4 hợp lệ cho {video_id}: {exact}")
    if exact:
        return exact[0], "kaggle-input-read-only"
    if candidates:
        sizes = [(str(path), path.stat().st_size) for path in candidates]
        raise RuntimeError(f"Attached MP4 sai kích thước cho {video_id}: {sizes}; expected {expected_size}")
    fetch_video(catalogue[video_id], local_video)
    return local_video, "http-range"

def local_paths(video_id: str) -> dict[str, Path]:
    root = WORK_DIR / MEMBER / video_id
    root.mkdir(parents=True, exist_ok=True)
    return {
        "root": root,
        "video": root / f"{video_id}.mp4",
        "mapping": root / f"{video_id}.csv",
        "shots": root / f"{video_id}.shots.jsonl",
        "report": root / f"{video_id}.report.json",
        "marker": root / f"{video_id}.complete.json",
    }

## Step 0d — preflight và resume

Cell này chỉ đọc marker/hash và kiểm một mapping đầu tiên; chưa tải video, chưa chạy GPU. Nếu member/worker sai thì dừng ở đây.

In [ ]:
completed = [video_id for video_id in assigned_videos if completion_marker(video_id) is not None]
pending = [video_id for video_id in assigned_videos if video_id not in set(completed)]
print({"assigned": len(assigned_videos), "complete": len(completed), "pending": len(pending), "first_pending": pending[:3]})
if pending:
    probe = WORK_DIR / "preflight" / f"{pending[0]}.csv"
    btc_transport.get(btc_layout.map_keyframes(pending[0]), probe)
    probe_fps, probe_rows = read_literal_fps(probe)
    print({"mapping_probe": pending[0], "literal_fps": probe_fps, "rows": probe_rows, "sha256": sha256_file(probe)})
    probe.unlink(missing_ok=True)

## Step 1 — chạy shard và upload

`PREFLIGHT` luôn chọn đúng một video pending ngắn nhất và chạy bằng model thật. Chỉ chuyển sang `SHARD` sau khi preflight PASS; mỗi phiên ngừng nhận video mới sau `SESSION_MAX_MINUTES` và lần Run All kế tiếp resume bằng marker. Model được giữ trong RAM, nhiều cửa sổ TransNet được batch trên GPU. Output shots và report được upload trước, đọc lại SHA-256, rồi marker mới được upload cuối cùng.

In [ ]:
def process_video(video_id: str) -> dict:
    started = time.perf_counter()
    stage_started = started
    producer_timings = {}
    local = local_paths(video_id)
    remote = remote_paths(video_id)
    try:
        print(f"{video_id}: fetch mapping", flush=True)
        btc_transport.get(btc_layout.map_keyframes(video_id), local["mapping"])
        literal_fps, mapping_rows = read_literal_fps(local["mapping"])
        mapping_sha = sha256_file(local["mapping"])
        producer_timings["mapping_fetch"] = time.perf_counter() - stage_started
        stage_started = time.perf_counter()
        print(f"{video_id}: fetch video", flush=True)
        video_path, source_mode = resolve_video_source(video_id, local["video"])
        producer_timings["video_fetch"] = time.perf_counter() - stage_started
        stage_started = time.perf_counter()
        print(f"{video_id}: hash video", flush=True)
        video_sha = sha256_file(video_path)
        producer_timings["video_hash"] = time.perf_counter() - stage_started

        progress_bucket = {"value": -1}
        def detector_progress(done: int, total: int) -> None:
            bucket = min(10, (10 * done) // max(total, 1))
            if bucket > progress_bucket["value"]:
                progress_bucket["value"] = bucket
                print(f"{video_id}: TransNet {done}/{total} windows ({bucket * 10}%)", flush=True)

        print(f"{video_id}: decode + TransNet", flush=True)
        proposals, detector_report = detect_video(
            video_path,
            repo=TRANSNET_DIR,
            weights=TRANSNET_WEIGHTS,
            literal_fps=literal_fps,
            threshold=float(tn["threshold"]),
            device=tn["device"],
            window_batch_size=INFERENCE_WINDOW_BATCH_SIZE,
            progress=detector_progress,
        )
        if any(row.video_id != video_id for row in proposals):
            raise RuntimeError(f"Detector trả sai video_id cho {video_id}")
        report = {
            **detector_report,
            "schema_version": 1,
            "run_id": CFG["run_id"],
            "member": MEMBER,
            "worker": WORKER,
            "video_id": video_id,
            "source_url": catalogue[video_id].url,
            "source_zip": catalogue[video_id].zip_name,
            "source_mode": source_mode,
            "source_path": str(video_path),
            "video_sha256": video_sha,
            "btc_mapping_remote": btc_layout.map_keyframes(video_id),
            "btc_mapping_sha256": mapping_sha,
            "btc_mapping_rows": mapping_rows,
            "repo_commit": REPO_COMMIT,
            "config_sha256": CONFIG_SHA256,
            "run_scope": RUN_SCOPE,
            "output_root": remote["shots"].rsplit("/shots/", 1)[0],
            "producer_timings_seconds": producer_timings,
            "session_max_minutes": SESSION_MAX_MINUTES,
            "measurement_command": "Kaggle notebook process_video(video_id)",
            "created_at_utc": utc_now(),
        }
        report["artifact_run_hash"] = sha256_json(report)
        atomic_write_jsonl(local["shots"], (row.model_dump(mode="json") for row in proposals))
        atomic_write_json(local["report"], report)
        local_shots_sha = sha256_file(local["shots"])
        local_report_sha = sha256_file(local["report"])

        upload_started = time.perf_counter()
        print(f"{video_id}: upload + verify", flush=True)
        output_transport.put(local["shots"], remote["shots"])
        output_transport.put(local["report"], remote["report"])
        remote_shots_sha = output_transport.sha256(remote["shots"])
        remote_report_sha = output_transport.sha256(remote["report"])
        if remote_shots_sha != local_shots_sha or remote_report_sha != local_report_sha:
            raise RuntimeError(f"Remote hash mismatch for {video_id}; không viết COMPLETE marker")

        marker = {
            "schema_version": 1,
            "status": "COMPLETE",
            "run_id": CFG["run_id"],
            "output_root": remote["shots"].rsplit("/shots/", 1)[0],
            "member": MEMBER,
            "worker": WORKER,
            "video_id": video_id,
            "repo_commit": REPO_COMMIT,
            "config_sha256": CONFIG_SHA256,
            "transnet_commit": actual_transnet_commit,
            "transnet_source_sha256": TRANSNET_SOURCE_SHA256,
            "transnet_weights_sha256": TRANSNET_WEIGHTS_SHA256,
            "threshold": float(tn["threshold"]),
            "device": tn["device"],
            "session_max_minutes": SESSION_MAX_MINUTES,
            "requested_inference_window_batch_size": INFERENCE_WINDOW_BATCH_SIZE,
            "inference_window_batch_size": detector_report["window_batch_size"],
            "model_cache_hit": detector_report["model_cache_hit"],
            "detector_timings_seconds": detector_report["timings_seconds"],
            "upload_verify_seconds": round(time.perf_counter() - upload_started, 3),
            "literal_fps": literal_fps,
            "video_sha256": video_sha,
            "btc_mapping_sha256": mapping_sha,
            "shots": remote["shots"],
            "shots_sha256": remote_shots_sha,
            "report": remote["report"],
            "report_sha256": remote_report_sha,
            "shot_count": len(proposals),
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "completed_at_utc": utc_now(),
        }
        marker["marker_payload_hash"] = sha256_json(marker)
        atomic_write_json(local["marker"], marker)
        output_transport.put(local["marker"], remote["marker"])
        if output_transport.sha256(remote["marker"]) != sha256_file(local["marker"]):
            raise RuntimeError(f"COMPLETE marker hash mismatch for {video_id}")
        return marker
    finally:
        shutil.rmtree(local["root"], ignore_errors=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def build_member_manifest(failures: list[dict]) -> dict:
    verified: list[dict] = []
    for video_id in assigned_videos:
        marker = completion_marker(video_id)
        if marker is not None:
            verified.append({"video_id": video_id, "marker_payload_hash": marker["marker_payload_hash"], "shots_sha256": marker["shots_sha256"], "report_sha256": marker["report_sha256"]})
    value = {
        "schema_version": 1,
        "run_id": CFG["run_id"],
        "member": MEMBER,
        "worker": WORKER,
        "repo_commit": REPO_COMMIT,
        "config_sha256": CONFIG_SHA256,
        "assigned_count": len(assigned_videos),
        "verified_complete_count": len(verified),
        "verified": verified,
        "failures_this_session": failures,
        "created_at_utc": utc_now(),
    }
    value["manifest_payload_hash"] = sha256_json(value)
    return value

STOP_ON_ERROR = False
pending_by_size = sorted(pending, key=lambda video_id: (catalogue[video_id].size, video_id))
todo = pending_by_size[:1] if RUN_SCOPE == "PREFLIGHT" else pending_by_size
session_started = time.perf_counter()
processed_bytes = 0
failures: list[dict] = []
for ordinal, video_id in enumerate(todo, 1):
    session_elapsed = time.perf_counter() - session_started
    if ordinal > 1 and session_elapsed >= SESSION_MAX_MINUTES * 60:
        print({"session_checkpoint": "TIME_BUDGET_REACHED", "elapsed_minutes": round(session_elapsed / 60, 2), "remaining": len(todo) - ordinal + 1}, flush=True)
        break
    print(f"[{ordinal}/{len(todo)}] {video_id} ({catalogue[video_id].size / 2**20:.1f} MiB)", flush=True)
    try:
        marker = process_video(video_id)
        processed_bytes += catalogue[video_id].size
        session_elapsed = time.perf_counter() - session_started
        remaining_bytes = sum(catalogue[item].size for item in todo[ordinal:])
        eta_hours = (remaining_bytes * session_elapsed / processed_bytes / 3600) if processed_bytes else None
        print({"complete": video_id, "shots": marker["shot_count"], "seconds": marker["elapsed_seconds"], "model_cache_hit": marker["model_cache_hit"], "eta_hours_estimate_this_todo": None if eta_hours is None else round(eta_hours, 2), "eta_label": "ESTIMATE_FROM_COMPLETED_BYTES"}, flush=True)
    except Exception as error:
        failure = {"video_id": video_id, "error_type": type(error).__name__, "error": str(error), "at_utc": utc_now()}
        failures.append(failure)
        print({"failed": failure})
        if STOP_ON_ERROR:
            raise

member_manifest = build_member_manifest(failures)
manifest_local = WORK_DIR / f"worker-manifest-{WORKER}.json"
manifest_remote = f"{CFG['output']['root'].rstrip('/')}/worker-manifests/{WORKER}.json"
atomic_write_json(manifest_local, member_manifest)
output_transport.put(manifest_local, manifest_remote)
if output_transport.sha256(manifest_remote) != sha256_file(manifest_local):
    raise RuntimeError("Member manifest upload hash mismatch")
print({"manifest": manifest_remote, "assigned": member_manifest["assigned_count"], "verified_complete": member_manifest["verified_complete_count"], "failures_this_session": len(failures), "sha256": sha256_file(manifest_local)})

## Step 2 — chốt phiên và bàn giao

Một worker chỉ hoàn tất khi `verified_complete_count == assigned_count`. Sau khi đủ 12 worker (hoặc đủ các worker được phép dùng), lead tải/merge toàn bộ `shots/*.jsonl`, kiểm trùng/missing video và downstream fingerprint, rồi mới chạy Gate1 một lần trên toàn corpus để tạo B1–B4 cùng hard delta budget. Không dùng temporal coverage để suy `NO_HEADROOM`; không tune bằng held-out.

Git chỉ cần push notebook + config + code. Video, weights và shot artifacts không đưa vào Git; chúng nằm trong BTC source/upstream chính thức và Modal Volume nghiên cứu có hash/provenance.

### Lỗi thường gặp

| dấu hiệu | cách xử lý |
|---|---|
| `Không thấy CUDA` | Chọn GPU trong Kaggle Settings, restart session rồi Run All |
| thiếu Modal token | Tạo đúng hai Kaggle Secrets; không dán token vào cell |
| clone repo private thất bại | Thêm `GITHUB_TOKEN` có quyền đọc repo |
| thiếu Git LFS/official weight | Không tải mirror; báo lead kèm nguyên lỗi cell upstream |
| BTC host bị block/chậm | Attach private Kaggle Dataset chứa đúng MP4 BTC; notebook ưu tiên đọc `/kaggle/input` read-only trước HTTP Range |
| một video lỗi hoặc session hết giờ | Run All lại; marker hợp lệ được skip, video lỗi được thử lại |
| `verified_complete < assigned` | Chưa xong shard; xem `failures_this_session` rồi rerun |

Khi báo lỗi cho lead, gửi `MEMBER`, `WORKER`, video ID, tên exception và output hash/commit đã in; tuyệt đối không gửi secret.